# E5: Fine Tuning BERT for Vulnerability Analysis

Fine tune `bert-base-cased` on `SourceCodeFLM.csv` to predict `vulnerability_type` from `code` and `language`.

Pipeline: input pre-processing -> fine tuning (training) -> evaluation (testing) with `accuracy`.

In [13]:
%pip install torch torchvision torchaudio numpy pandas nltk transformers scikit-learn 'accelerate>=1.1.0' evaluate datasets


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import numpy as np
import pandas as pd
import re
import torch
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from datasets import Dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer
from transformers import set_seed
import evaluate

[nltk_data] Downloading package stopwords to /Users/admin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [15]:
DATA_FILE  = 'SourceCodeFLM.csv'
MODEL_NAME = 'bert-base-cased'
NUM_EPOCHS = 5
MAX_LENGTH = 256
SEED       = 1
set_seed(SEED)
device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: mps


## 1. Load the full dataset

In [16]:
df = pd.read_csv(DATA_FILE)
df = df.dropna(subset=['vulnerability_type', 'code', 'language']).reset_index(drop=True)
print(df.shape)
print(df['vulnerability_type'].value_counts())
print(df['language'].value_counts())

(4649, 5)
vulnerability_type
Clear                         2308
Buffer Overflow                456
Command Injection              309
SQL Injection                  264
Memory Corruption              251
Cross-Site Scripting (XSS)     191
Insecure Deserialization       146
Out-of-Bounds                  136
Path Traversal                 128
Hardcoded Secrets              115
SSRF                           102
File Inclusion                  99
XSS                             90
Code Injection                  54
Name: count, dtype: int64
language
java          1071
c             1055
php            725
python         666
javascript     577
cpp            555
Name: count, dtype: int64


## 2. Input pre-processing

Remove punctuation, stop words, short tokens, URLs and emojis from the source code, then prefix the language so the model sees both independent variables.

In [17]:
sw = set(stopwords.words('english'))
punctuations = '''!()-[]{};:'"\\,<>./?@#$%^&*_~=+|`'''
emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"
                           u"\U0001F300-\U0001F5FF"
                           u"\U0001F680-\U0001F6FF"
                           u"\U0001F1E0-\U0001F1FF"
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)

def preprocessText(text):
    if isinstance(text, str):
        text = re.sub(r'https?://\S+', ' ', text)
        for p in punctuations:
            text = text.replace(p, ' ')
        text = [word.lower() for word in text.split() if word.lower() not in sw]
        text = [word for word in text if len(word) >= 3]
        text = ' '.join(text)
        text = emoji_pattern.sub(r'', text)
    else:
        text = ''
    return text

df['CLEANEDCODE'] = df['language'] + ' ' + df['code'].apply(preprocessText)
df[['language', 'vulnerability_type', 'CLEANEDCODE']].head()

,language,vulnerability_type,CLEANEDCODE
0,cpp,Out-of-Bounds,cpp include iostream class unsafeclass int dat...
1,java,Clear,java import java util scanner public class sec...
2,python,Clear,python import subprocess def safe ping host im...
3,java,Path Traversal,java import java file import java fileinputstr...
4,c,Clear,c include stdio include stdlib void safe file ...


## 3. Encode the target class and split 70% train / 30% test

In [18]:
labelNames = sorted(df['vulnerability_type'].unique())
labelDict  = {name: idx for idx, name in enumerate(labelNames)}
id2label   = {idx: name for name, idx in labelDict.items()}
df['CATEG'] = df['vulnerability_type'].map(labelDict)
print('Vulnerability types:', labelDict)

train_df, test_df = train_test_split(df, test_size=0.3, random_state=SEED, stratify=df['CATEG'])
train_df = train_df[['CLEANEDCODE', 'CATEG']].reset_index(drop=True)
test_df  = test_df[['CLEANEDCODE', 'CATEG']].reset_index(drop=True)
print('Train:', train_df.shape, 'Test:', test_df.shape)

Vulnerability types: {'Buffer Overflow': 0, 'Clear': 1, 'Code Injection': 2, 'Command Injection': 3, 'Cross-Site Scripting (XSS)': 4, 'File Inclusion': 5, 'Hardcoded Secrets': 6, 'Insecure Deserialization': 7, 'Memory Corruption': 8, 'Out-of-Bounds': 9, 'Path Traversal': 10, 'SQL Injection': 11, 'SSRF': 12, 'XSS': 13}
Train: (3254, 2) Test: (1395, 2)


In [19]:
trainDataset = Dataset.from_pandas(train_df)
testDataset  = Dataset.from_pandas(test_df)

## 4. Tokenize

In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(dataset_):
    return tokenizer(dataset_['CLEANEDCODE'], padding='max_length', truncation=True, max_length=MAX_LENGTH)

tokenized_train = trainDataset.map(tokenize_function, batched=True)
tokenized_test  = testDataset.map(tokenize_function, batched=True)
tokenized_train = tokenized_train.rename_column('CATEG', 'labels')
tokenized_test  = tokenized_test.rename_column('CATEG', 'labels')

Map: 100%|██████████| 1395/1395 [00:00<00:00, 11080.85 examples/s]


## 5. Load the pre-trained model

`BertForSequenceClassification` adds a new classifier head on top of BERT. Its weights are not in the checkpoint, so they are reported as missing. They are saved to `missing.txt`.

In [21]:
model, loading_info = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labelDict),
    id2label=id2label,
    label2id=labelDict,
    output_loading_info=True,
)
print(type(model).__name__)
print('Missing keys:', loading_info['missing_keys'])
print('Unexpected keys:', loading_info['unexpected_keys'])

with open('missing.txt', 'w') as f:
    f.write(f'Missing keys when loading {type(model).__name__} from {MODEL_NAME}:\n')
    for key in sorted(loading_info['missing_keys']):
        f.write(f'{key}\n')
    f.write('\nThese classifier head weights are not in the pre-trained checkpoint, so they are newly initialized and learned during fine tuning.\n')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6689.75it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint

BertForSequenceClassification
Missing keys: {'classifier.bias', 'classifier.weight'}
Unexpected keys: {'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias'}


## 6. Fine tune

In [22]:
training_args = TrainingArguments(
    output_dir                  = 'bert4vpm-output',
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    logging_strategy            = 'epoch',
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    optim                       = 'adamw_torch',
    learning_rate               = 3e-5,
    weight_decay                = 0.01,
    warmup_steps                = 100,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'accuracy',
    save_total_limit            = 1,
    report_to                   = 'none',
    seed                        = SEED,
)

In [23]:
metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [24]:
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized_train,
    eval_dataset    = tokenized_test,
    compute_metrics = compute_metrics,
)
trainer.train()

/Users/admin/Documents/coding_stuff/COMP-5700/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,1.583874,0.698050,0.853763
2,0.495701,0.440960,0.866667
3,0.311473,0.519259,0.848746
4,0.224309,0.539133,0.832975
5,0.156297,0.621914,0.835125


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.27it/s]
/Users/admin/Documents/coding_stuff/COMP-5700/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.28it/s]
/Users/admin/Documents/coding_stuff/COMP-5700/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.38it/s]
/Users/admin/Documents/coding_stuff/COMP-5700/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model sh

TrainOutput(global_step=1020, training_loss=0.5543306649899951, metrics={'train_runtime': 718.3201, 'train_samples_per_second': 22.65, 'train_steps_per_second': 1.42, 'total_flos': 2140639049717760.0, 'train_loss': 0.5543306649899951, 'epoch': 5.0})

## 7. Evaluate on the 30% test set

In [25]:
eval_results = trainer.evaluate()
print(eval_results)

pred_output = trainer.predict(tokenized_test)
y_pred = np.argmax(pred_output.predictions, axis=-1)
y_true = pred_output.label_ids
test_accuracy = accuracy_score(y_true, y_pred)
print('Test accuracy:', round(test_accuracy, 4))
report = classification_report(y_true, y_pred, labels=list(id2label.keys()), target_names=labelNames, zero_division=0)
print(report)

/Users/admin/Documents/coding_stuff/COMP-5700/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.156297,0.440698,5,0.866667


{'eval_loss': 0.4406978189945221, 'eval_accuracy': 0.8666666666666667}


/Users/admin/Documents/coding_stuff/COMP-5700/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Test accuracy: 0.8667
                            precision    recall  f1-score   support

           Buffer Overflow       0.99      0.93      0.96       137
                     Clear       0.82      0.97      0.89       693
            Code Injection       1.00      0.88      0.93        16
         Command Injection       0.93      0.85      0.89        93
Cross-Site Scripting (XSS)       0.98      0.95      0.96        57
            File Inclusion       0.50      0.37      0.42        30
         Hardcoded Secrets       1.00      0.76      0.87        34
  Insecure Deserialization       0.96      0.55      0.70        44
         Memory Corruption       1.00      0.45      0.62        75
             Out-of-Bounds       1.00      0.37      0.54        41
            Path Traversal       1.00      0.92      0.96        38
             SQL Injection       0.96      0.97      0.97        79
                      SSRF       0.56      0.48      0.52        31
                       XS

In [26]:
epoch_log = [(int(round(e['epoch'])), e['eval_accuracy'], e['eval_loss']) for e in trainer.state.log_history if 'eval_accuracy' in e][:NUM_EPOCHS]
train_log = {int(round(e['epoch'])): e['loss'] for e in trainer.state.log_history if 'loss' in e}
best_epoch, best_acc, _ = max(epoch_log, key=lambda row: row[1])

lines = [
    'Fine Tuned BERT for Vulnerability Analysis',
    f'Model: {MODEL_NAME} ({type(model).__name__})',
    'Dataset: SourceCodeFLM.csv (full dataset)',
    'Dependent variable: vulnerability_type',
    'Independent variables: code, language',
    f'Train/test split: 70/30 ({len(train_df)} train, {len(test_df)} test)',
    'Metric: accuracy',
    '',
    'Epoch | Training Loss | Validation Loss | Accuracy',
]
for epoch, acc, loss in epoch_log:
    lines.append(f'{epoch:>5} | {train_log.get(epoch, float("nan")):>13.4f} | {loss:>15.4f} | {acc:.4f}')
lines += [
    '',
    f'Number of epochs run: {NUM_EPOCHS}',
    f'Number of epochs needed to reach best accuracy: {best_epoch}',
    f'Final test accuracy (best model): {test_accuracy:.4f}',
    '',
    'Per-class results on the test set:',
    report,
]
with open('output.txt', 'w') as f:
    f.write('\n'.join(lines))
print('\n'.join(lines))

Fine Tuned BERT for Vulnerability Analysis
Model: bert-base-cased (BertForSequenceClassification)
Dataset: SourceCodeFLM.csv (full dataset)
Dependent variable: vulnerability_type
Independent variables: code, language
Train/test split: 70/30 (3254 train, 1395 test)
Metric: accuracy

Epoch | Training Loss | Validation Loss | Accuracy
    1 |        1.5839 |          0.6980 | 0.8538
    2 |        0.4957 |          0.4410 | 0.8667
    3 |        0.3115 |          0.5193 | 0.8487
    4 |        0.2243 |          0.5391 | 0.8330
    5 |        0.1563 |          0.6219 | 0.8351

Number of epochs run: 5
Number of epochs needed to reach best accuracy: 2
Final test accuracy (best model): 0.8667

Per-class results on the test set:
                            precision    recall  f1-score   support

           Buffer Overflow       0.99      0.93      0.96       137
                     Clear       0.82      0.97      0.89       693
            Code Injection       1.00      0.88      0.93       